# Assignment: Event studies, matching and synthetic control
**Econ 521 · Lab assignment · Released Fri 13 Nov 2026 · Due Fri 27 Nov 2026, 11:59 PM (Canvas)**

This assignment has **four applied questions worth 25 points each**. Three use the Basque data (Questions 1, 3 and 4) and one uses LaLonde (Question 2). They build on the two Friday labs, but none of the analyses here was done in a lab. The labs rebuilt Mahalanobis matching and the paper's own synthetic control. Here you work with propensity scores, an event study, and a synthetic control **you** design and test.

**What to submit:** this notebook, fully executed (Kernel → Restart & Run All), uploaded to Canvas. Written answers go in the markdown cells marked **Answer**. Stay within word limits; short, precise answers score best.

**Data:** `lalonde.csv` and `basque.csv`, the same files as in the labs, in the same folder as this notebook.

**Code you may reuse:** any function from the two labs. The setup cell already defines `smd()`, `solve_w()` and `weighted_att_se()` for you.

| Question | Topic | Data | Points |
|---|---|---|---|
| 1 | An event study of the Basque conflict | `basque.csv` | 25 |
| 2 | Propensity scores, overlap and a placebo outcome | `lalonde.csv` | 25 |
| 3 | Design your own synthetic control | `basque.csv` | 25 |
| 4 | Is it real? Inference, robustness, and two designs side by side | `basque.csv` | 25 |

**How you are graded.** Correct implementation counts, and so does honest interpretation. In Questions 3 and 4 there is no single right answer. You are graded on whether you **state your design before you see results**, justify it with the lecture and the paper, implement it correctly, and report what you find, including results that do not support a clean story.

In [ ]:
# !pip install numpy pandas scipy statsmodels matplotlib   # uncomment if needed
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.optimize import minimize
import warnings; warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
BLUE, ORANGE, GREY, GREEN, PURPLE, RED = "#2b6cb0", "#dd6b20", "#a0aec0", "#2f855a", "#6b46c1", "#c53030"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
here = lambda f: Path(f) if Path(f).exists() else Path("data") / f

# ---------- LaLonde ----------
ld = pd.read_csv(here("lalonde.csv"))
for r in ["black", "hispan", "white"]:
    ld[r] = (ld["race"] == r).astype(int)
BENCHMARK = 1794

# ---------- Basque ----------
NAMES = {"Basque Country (Pais Vasco)": "Basque Country", "Cataluna": "Catalonia", "Madrid (Comunidad De)": "Madrid",
         "Baleares (Islas)": "Balearic Islands", "Rioja (La)": "La Rioja", "Navarra (Comunidad Foral De)": "Navarre",
         "Principado De Asturias": "Asturias", "Murcia (Region de)": "Murcia", "Comunidad Valenciana": "Valencia",
         "Castilla Y Leon": "Castile and Leon", "Castilla-La Mancha": "Castile-La Mancha", "Andalucia": "Andalusia",
         "Canarias": "Canary Islands", "Cantabria": "Cantabria", "Extremadura": "Extremadura", "Galicia": "Galicia",
         "Aragon": "Aragon", "Spain (Espana)": "Spain"}
bq = pd.read_csv(here("basque.csv"))
bq["region"] = bq["regionname"].map(NAMES)
bq = bq[bq["region"] != "Spain"].copy()                          # 17 regions
Y = bq.pivot(index="year", columns="region", values="gdpcap")    # GDP per capita, thousands of 1986 USD
TREATED = "Basque Country"
OTHERS = [r for r in Y.columns if r != TREATED]

# ---------- helpers from the labs ----------
def smd(x_t, x_c, w_c=None):
    '''Standardized mean difference with the treated SD in the denominator (comparison units may be weighted).'''
    w_c = np.ones(len(x_c)) if w_c is None else np.asarray(w_c)
    return (np.mean(x_t) - np.average(x_c, weights=w_c)) / np.std(x_t, ddof=1)

def weighted_att_se(y_t, y_c, w_c):
    '''ATT and HC1 robust SE from a weighted regression of y on a treatment dummy (treated weight 1).'''
    w_c = np.asarray(w_c, float); used = w_c > 0
    y = np.r_[y_t, np.asarray(y_c)[used]]
    d = np.r_[np.ones(len(y_t)), np.zeros(used.sum())]
    wt = np.r_[np.ones(len(y_t)), w_c[used] / w_c[used].mean()]
    fit = sm.WLS(y, sm.add_constant(d), weights=wt).fit(cov_type="HC1")
    return fit.params[1], fit.bse[1]

def solve_w(v, X1s, X0s):
    '''Synthetic control weights for a diagonal V = diag(v), predictors already standardized.
    X1s: (K,) treated predictors; X0s: (K, J) donor predictors.'''
    J = X0s.shape[1]
    A_, a_ = X0s * np.sqrt(v)[:, None], X1s * np.sqrt(v)
    H, g = A_.T @ A_, A_.T @ a_
    res = minimize(lambda w: w @ H @ w - 2 * g @ w, np.full(J, 1 / J), jac=lambda w: 2 * H @ w - 2 * g,
                   method="SLSQP", bounds=[(0, 1)] * J,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(J)}],
                   options={"ftol": 1e-14, "maxiter": 1000})
    w = np.clip(res.x, 0, None)
    return w / w.sum()

print(f"LaLonde: {len(ld)} people | Basque panel: {Y.shape[1]} regions, {Y.index.min()}-{Y.index.max()}")

---
## Question 1 · An event study of the Basque conflict (25 points)

Before reaching for synthetic control, most applied economists would run an event study. Do that here and see where it leads.

**Setup.** Use the 17-region Basque panel (Spain as a whole is already dropped). The outcome is **log** GDP per capita. The treated unit is the Basque Country and the reference year is **1969**, the last year of the pre-terrorism decade Abadie and Gardeazabal use.

**(a) (5 pts)** Estimate by OLS
$$\ln y_{it} = \alpha_i + \lambda_t + \sum_{k\neq 1969}\beta_k\,\mathbf 1[i=\text{Basque}]\,\mathbf 1[t=k] + \varepsilon_{it},$$
with region and year fixed effects and standard errors clustered by region. Plot $\hat\beta_k$ with 95% confidence intervals for 1955–1997. Report the average of $\hat\beta_k$ over 1975–1997 and convert it to a percent effect, $100\,(e^{\bar\beta}-1)$.

In [ ]:
# Your code here

**(b) (4 pts)** Show numerically that every $\hat\beta_k$ equals a double difference you compute by hand from the log GDP panel: the Basque change from 1969 to year $k$ minus the average change of the other 16 regions. Then, in one sentence: **what weights does this event study put on the 16 comparison regions?** Contrast them with the synthetic control weights from the lab.

In [ ]:
# Your code here

**Answer (b):**

**(c) (6 pts) Pre-trends.**
(i) Describe the pre-period coefficients (1955–1968). Is the parallel trends assumption plausible?
(ii) Fit a straight line through the pre-period coefficients (1955–1969, including $\hat\beta_{1969}=0$), extrapolate it to 1970–1997, and report the **trend-adjusted** average effect over 1975–1997 (in log points and in percent).
(iii) In ≤ 100 words: what assumption replaces parallel trends once you do this, and why might extrapolating a 15-year trend over 28 years be risky in this application?

In [ ]:
# Your code here

**Answer (c):**

**(d) (6 pts) Inference with one treated unit.**
(i) Look at the clustered standard errors of the pre-period coefficients. Explain why they are not credible here. *Hint: with a full set of Basque × year dummies, what are the Basque residuals? So where does all the estimated variability come from?*
(ii) **Placebo in space.** For each of the 16 other regions, pretend it was the treated region (its comparison group is the 15 remaining non-Basque regions) and compute the same 1975–1997 average effect. Report the Basque rank (most negative = rank 1) and the permutation p-value. What is the smallest p-value this test can ever produce?
(iii) Plot each region's placebo effect against its average GDP per capita in 1960–69, with the Basque Country included and labelled. What does the pattern tell you about the parallel trends assumption in (a)?

In [ ]:
# Your code here

**Answer (d):**

**(e) (4 pts) A hand-picked comparison group.** Choose 2–4 comparison regions with a rule you **state before** looking at any post-1969 data (for example, the richest regions in the 1960s, or the most industrial). Re-compute the average 1975–1997 effect against only those regions, and also report the average pre-period (1955–1968) coefficient. In ≤ 80 words, relate the result to the **two problems with hand-picked comparison groups** from the lecture (Card's Mariel boatlift study).

In [ ]:
# Your code here

**Answer (e):**

---
## Question 2 · Propensity scores, overlap and a placebo outcome (25 points)

LaLonde data again. The lab matched on a Mahalanobis distance. Here you match and weight on the **propensity score** (Thursday 12 Nov lecture). Create the unemployment indicators `u74 = 1[re74 = 0]` and `u75 = 1[re75 = 0]` first.

**(a) (5 pts)** Estimate the propensity score with a logit. Your specification **must** include `age, educ, black, hispan, married, nodegree, re74, re75, u74, u75` and **at least one** nonlinear term (a square or an interaction). Justify the nonlinear term(s) in one sentence. Plot the distribution of the estimated **log-odds** of the propensity score by group. How many trainees have a propensity score **above the largest** score in the PSID group, and how many PSID men have a score **below the smallest** score among trainees? Describe in one sentence what kind of trainee has no comparable PSID man.

In [ ]:
# Your code here

**Answer (a):**

**(b) (4 pts)** Trim the sample to the region of common support, $[\max(\min_T \hat p, \min_C \hat p),\ \min(\max_T \hat p, \max_C \hat p)]$. Report how many trainees and PSID men you drop. In ≤ 60 words: **what is your estimand after trimming**, and how does it differ from the ATT for all 185 trainees?

In [ ]:
# Your code here

**Answer (b):**

**(c) (8 pts)** On the trimmed sample:
1. **Matching.** 1-nearest-neighbour matching on the **log-odds** of the propensity score, with replacement, with a **caliper of 0.2 pooled standard deviations** of the log-odds, $0.2\sqrt{(s_T^2+s_C^2)/2}$ (Austin 2011). Average over ties. Report the number of matched trainees, the ATT, its robust SE (use `weighted_att_se`), and a balance table (SMDs) for `age, educ, black, hispan, white, married, nodegree, re74, re75, u74, u75`.
2. **Inverse probability weighting.** Estimate the ATT by weighting each PSID man by $\hat p/(1-\hat p)$ (trainees get weight 1). Report the ATT and the same balance table under these weights.
3. In ≤ 80 words, compare the two estimates with each other and with the experimental benchmark.

In [ ]:
# Your code here

**Answer (c):**

**(d) (5 pts) A placebo outcome** (Imbens 2015). Earnings in 1975 were determined before the program. Re-estimate the propensity score **without `re75` and `u75`** (drop them from every term), redo the trimming and the PS matching from (b)–(c), and use **`re75` as the outcome**. If conditioning on the remaining covariates removes selection, the estimated "effect" on 1975 earnings should be close to zero. Report the estimate and its SE. In ≤ 80 words, interpret the result, including what the SE implies about the **power** of this test.

In [ ]:
# Your code here

**Answer (d):**

**(e) (3 pts) Sensitivity.** Report the PS-matching and IPW estimates (trimmed, as in (b)–(c)) for at least **one other** propensity-score specification, for example the linear one without `u74`, `u75` or nonlinear terms. Include the largest |SMD| after matching. In ≤ 120 words: **what would you tell a policymaker** about whether the PSID comparison group can recover the effect of the NSW program?

In [ ]:
# Your code here

**Answer (e):**

---
## Question 3 · Design your own synthetic control (25 points)

In the lab you reproduced the paper's synthetic Basque Country. Now **you** make the design choices.

**(a) (6 pts) Design memo, written before you estimate anything (≤ 200 words).** State and justify:
- **The first treated year $T_0$.** Use Table 1 of the paper (ETA killings and kidnappings by year). Refer to the lecture's *no anticipation* requirement and to the bias bound (the number of pre-intervention periods).
- **The donor pool.** Do you exclude any region? Think about **interference** (where else did ETA operate? footnote 5 of the paper says where Basques live outside the Basque Country) and **large idiosyncratic shocks** to a donor during the study period.
- **The predictors**, in two groups: **(O)** lagged values of GDP per capita, and **(S)** structural growth predictors. Mind data availability: sectoral shares exist only for odd years 1961–69, schooling for 1964–69, investment for 1964–95, and population density only for 1969.

Your specification must differ from the lab's in at least one substantive way (timing, donor pool, or predictors), beyond the choice of $V$ in (b).

**Answer (a), design memo:**

**(b) (8 pts) Choose $V$ by cross-validation** (lecture slide 40; Abadie, Diamond & Hainmueller 2015). With 16 donors, a free search over every diagonal element of $V$ is fragile. It tends to put all the weight on one predictor, and then $W$ is no longer unique. So restrict $V$ to a **one-parameter family**. After standardizing every predictor by its SD across units, each lagged-outcome predictor gets weight $\alpha/n_O$ and each structural predictor $(1-\alpha)/n_S$, with $\alpha\in[0,1]$.
1. Split your pre-period into a **training** window and a **validation** window.
2. For each $\alpha$ on a grid (0, 0.05, …, 1), compute $W(\alpha)$ using lagged outcomes **from the training window only** (plus the structural predictors), and record the mean squared prediction error of GDP per capita over the **validation** window.
3. Pick $\alpha^*$. Then recompute $W^*$ with lagged outcomes from the **whole** pre-period.

Plot validation MSPE against $\alpha$. Report $\alpha^*$ and $W^*$. In ≤ 80 words: what does $\alpha^*$ tell you about the relative value of matching the outcome path against matching the economic structure here? Remember the lab's Appendix B comparison.

*Data note:* the structural predictors are only observed in the 1960s (density only in 1969), so they partly overlap your validation window. Treat them as fixed characteristics and say in your answer that the validation is therefore not fully out of sample.

*(Optional, no extra points: try a free search over all diagonal elements of $V$ from several starting values and describe what happens.)*

In [ ]:
# Your code here

**Answer (b):**

**(c) (6 pts) Results.** Report: (i) the weights; (ii) a predictor balance table (Basque Country, your synthetic, average of your donors); (iii) the pre-period RMSPE; (iv) a figure with the actual and synthetic GDP paths and a figure with the gap as a percent of Basque GDP per capita; (v) the average gap over your post-period and over 1980–1997 (to compare with the lab's −10.7%).

In [ ]:
# Your code here

**(d) (5 pts)** In ≤ 200 words: how did your design choices change the weights relative to the paper's (Catalonia 0.8508, Madrid 0.1492)? Why does your estimate differ from the paper's, or not?

In [ ]:
# (Optional) code supporting your answer to (d)

**Answer (d):**

---
## Question 4 · Is it real? Inference, robustness, and two designs side by side (25 points)

Use **your Q3 procedure** throughout: same predictors, same cross-validation of $\alpha$, same timing.

**(a) (8 pts) In-space placebos** (Abadie, Diamond & Hainmueller 2010; lecture slides 31–33). Apply your full procedure, **including re-choosing $\alpha$ by cross-validation**, to each donor region in turn. Its donor pool is your other donors; never include the Basque Country.
1. Plot all placebo gaps (in thousands of USD) in grey, with the Basque gap highlighted.
2. Compute each unit's post/pre RMSPE ratio. Report the Basque rank and the permutation p-value.
3. Repeat (2) after dropping placebos whose pre-period RMSPE is more than **5 times** the Basque one.
4. In ≤ 120 words: what do you conclude? Why does the Basque Country's position at the **edge of the convex hull** matter for this test, and what do the placebo regions with the largest ratios have in common?

In [ ]:
# Your code here

**Answer (a):**

**(b) (5 pts) In-time placebo.** Pretend the conflict started in **1965** (or at least 5 years before your $T_0$). Re-fit your synthetic control using **lagged outcomes only from before the placebo date**, keeping your $\alpha^*$, and structural predictors measured as early as the data allow. Plot the gap from 1955 to 1980. Does a "gap" open between the placebo date and your real $T_0$? What does its size tell you about the noise in your counterfactual?

In [ ]:
# Your code here

**Answer (b):**

**(c) (5 pts) Leave-one-out.** For each donor with positive weight in $W^*$, drop it from the pool, re-run your full procedure (including cross-validation), and plot the leave-one-out synthetic paths with your original synthetic and the actual Basque path. Report the range of the average post-period gap.

In [ ]:
# Your code here

**Answer (c):**

**(d) (7 pts) Two designs, one figure.** On one figure plot: (i) your synthetic control gap (%); (ii) the Q1 event-study effects converted to percent, $100(e^{\hat\beta_k}-1)$, **with and without** the trend adjustment; (iii) the paper's synthetic control gap from the lab (0.8508 × Catalonia + 0.1492 × Madrid). Then write a conclusion (≤ 300 words) giving **your best estimate of the economic cost of the conflict and the main threat to it**, and explaining **why the event study and synthetic control disagree**. Refer to (1) the weights each design puts on the regions, (2) your figure from Q1(d)(iii), and (3) the lecture's factor model, $Y^N_{jt} = \delta_t + \theta_t Z_j + \lambda_t\mu_j + \varepsilon_{jt}$.

In [ ]:
# Your code here

**Answer (d), conclusion:**

---
*End of assignment. Before submitting: Kernel → Restart & Run All, and check that every figure and table appears.*